# Clean JSON Export Inspection

This notebook loads the latest `processed_export_*.json` from the minimal schema
and builds a quick catalog of the available tables and images grouped by section.

In [25]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

# Import the export reader library
from scraper.export_reader import load_latest_export

# Load the latest export
reader = load_latest_export()

# Print summary
reader.print_summary()

# Get documents dataframe
df_documents = reader.get_documents_dataframe()
print(f"\nLoaded {len(df_documents)} documents")


Loading latest export: processed_export_1779284288.json
EXPORT SUMMARY
Total documents: 13
Export file: /home/jgrzyb/Documents/Python/ai4es-oa-paper-scrapper/notebooks/paper_pipeline_data/exports/processed_export_1779284288.json

Documents by source:
source
pmc    13

Total tables: 66
Total images: 116

Document sections:
  Min sections: 10
  Max sections: 44
  Avg sections: 18.6


Loaded 13 documents


## Data Parsing

In [26]:
# Get all tables as a dataframe
df_tables = reader.get_all_tables_dataframe()
print(f"Total tables: {len(df_tables)}")
display(df_tables.head(20))

# Get all images as a dataframe
df_images = reader.get_all_images_dataframe()
print(f"\nTotal images: {len(df_images)}")
display(df_images.head(20))


Total tables: 66


,paper_id,section,table_index,global_index,csv_path
0,PMC10420062,Results,0,0,tables/PMC10420062/table_0.csv
1,PMC10420062,Results,1,1,tables/PMC10420062/table_1.csv
2,PMC10420062,Results,2,2,tables/PMC10420062/table_2.csv
3,PMC10420062,Results,3,3,tables/PMC10420062/table_3.csv
4,PMC10420062,Sensitivity analysis,0,4,tables/PMC10420062/table_4.csv
5,PMC10420062,Quality and bias risk assessment,0,5,tables/PMC10420062/table_5.csv
6,PMC10420062,Quality and bias risk assessment,1,6,tables/PMC10420062/table_6.csv
7,PMC9640961,CAUSAL RISK FACTORS FOR ASTHMA AND RESPIRATORY...,0,0,tables/PMC9640961/table_0.csv
8,PMC9640961,CAUSAL RISK FACTORS FOR ASTHMA AND RESPIRATORY...,1,1,tables/PMC9640961/table_1.csv
9,PMC9640961,CAUSAL RISK FACTORS FOR ASTHMA AND RESPIRATORY...,2,2,tables/PMC9640961/table_2.csv



Total images: 116


,paper_id,section,placeholder,caption,path
0,PMC10420062,Results,PMC_FIG_0,Figure 1 PRISMA 2020 flow diagram.,png/PMC10420062/falgy-04-1211949-g002.gif
1,PMC10420062,Results,PMC_FIG_1,Figure 2 Meta-analysis using a random effects ...,png/PMC10420062/falgy-04-1211949-g001.gif
2,PMC10420062,Results,PMC_FIG_2,Figure 3 Meta-analysis using a random effects ...,png/PMC10420062/falgy-04-1211949-g002.jpg
3,PMC10420062,Pooled estimate analysis of PEF measurements,PMC_FIG_3,Figure 2 Meta-analysis using a random effects ...,png/PMC10420062/falgy-04-1211949-g003.gif
4,PMC10420062,Pooled estimate analysis of PEF measurements,PMC_FIG_4,Figure 3 Meta-analysis using a random effects ...,png/PMC10420062/falgy-04-1211949-g003.jpg
5,PMC9640961,RESULTS,PMC_FIG_0,FIGURE 1 Preferred Reporting Items of Systemat...,png/PMC9640961/CLT2-12-e12207-g001.gif
6,PMC9640961,CAUSAL RISK FACTORS FOR ASTHMA AND RESPIRATORY...,PMC_FIG_1,FIGURE 2 Overview of causal effects of risk fa...,png/PMC9640961/CLT2-12-e12207-g002.gif
7,PMC9640961,Anthropometry,PMC_FIG_2,FIGURE 2 Overview of causal effects of risk fa...,png/PMC9640961/CLT2-12-e12207-g002.jpg
8,PMC10848175,RESULTS,PMC_FIG_0,FIGURE 1 PRISMA flow diagram for systematic re...,png/PMC10848175/CLT2-14-e12338-g007.gif
9,PMC10848175,RESULTS,PMC_FIG_1,FIGURE 2 Lifetime prevalence of self‐reported ...,png/PMC10848175/CLT2-14-e12338-g008.gif


## Utilities for inspection

In [27]:
def show_table(index):
    """Display a specific table from the tables dataframe."""
    if index < 0 or index >= len(df_tables):
        print('Index out of range')
        return
    
    row = df_tables.iloc[index]
    print(f"Paper: {row['paper_id']}")
    print(f"Section: {row['section']}")
    print(f"Table Index: {row['table_index']} (Global: {row['global_index']})")
    print(f"CSV Path: {row['csv_path']}")
    print()
    
    try:
        df = reader.load_table_csv(row['csv_path'])
        display(df)
    except FileNotFoundError as e:
        print(f"Error: {e}")

def show_image(index):
    """Display a specific image from the images dataframe."""
    if index < 0 or index >= len(df_images):
        print('Index out of range')
        return
    
    row = df_images.iloc[index]
    print(f"Paper: {row['paper_id']}")
    print(f"Section: {row['section']}")
    print(f"Caption: {row['caption']}")
    print(f"Placeholder: {row['placeholder']}")
    print()
    
    try:
        img_path = reader.get_file_path(row['path'])
        
        # Handle remote URLs (HTTP/HTTPS)
        if img_path.startswith('http://') or img_path.startswith('https://'):
            print(f"Remote image: {img_path}")
            display(Image(url=img_path))
        # Handle local files
        elif Path(img_path).exists():
            display(Image(filename=img_path))
        else:
            print(f"Image file not found: {img_path}")
    except Exception as e:
        print(f"Error displaying image: {e}")

def show_document(paper_id):
    """Display complete information about a document."""
    meta = reader.get_document_metadata(paper_id)
    if meta is None:
        print(f"Document not found: {paper_id}")
        return
    
    print(f"Paper ID: {meta['paper_id']}")
    print(f"Source: {meta['source']}")
    if meta['pmcid']:
        print(f"PMCID: {meta['pmcid']}")
    if meta['arxiv_id']:
        print(f"arXiv ID: {meta['arxiv_id']}")
    print(f"Authors: {', '.join(meta['authors'][:3])}{'...' if len(meta['authors']) > 3 else ''}")
    print(f"Emails: {len(meta['emails'])} found")
    print(f"Files: MD={bool(meta['md_path'])}, HTML={bool(meta['html_path'])}, PDF={bool(meta['pdf_path'])}")
    print(f"Content: {meta['num_sections']} sections, {meta['num_tables']} tables, {meta['num_images']} images")
    print()

def show_document_sections(paper_id):
    """Display all sections and their content for a document."""
    sections = reader.load_document_sections(paper_id)
    if not sections:
        print(f"Document not found: {paper_id}")
        return
    
    for i, sec in enumerate(sections, 1):
        print(f"\n{'='*60}")
        print(f"Section {i}: {sec['heading']}")
        print(f"{'='*60}")
        
        if sec['tables']:
            print(f"\n📊 Tables ({len(sec['tables'])}):")
            for j, tbl in enumerate(sec['tables'], 1):
                print(f"\n  Table {j} (index: {tbl['table_index']}):")
                if tbl['data'] is not None:
                    display(tbl['data'])
                else:
                    print(f"  ❌ {tbl.get('error', 'Could not load data')}")
        
        if sec['images']:
            print(f"\n🖼️  Images ({len(sec['images'])}):")
            for img in sec['images']:
                print(f"  - {img['caption']} ({img['placeholder']})")

In [28]:
# Example: Show first table
if len(df_tables) > 0:
    print("Example: First table in export:")
    show_table(0)
else:
    print("No tables found in export")


Example: First table in export:
Paper: PMC10420062
Section: Results
Table Index: 0 (Global: 0)
CSV Path: tables/PMC10420062/table_0.csv



,Study,Design,Objective,Outcomes,Patients,Treatment,Result
0,"Green (11), 1992, California, United States",RCT,I.v. MgSO4 as adjunct to standard therapy,Hospital admission and PEF.,120 patients (F: 77%). MgSO4: 58. Control: 62....,"SoC: nebulized albuterol, i.v. steroids, oxyge...",No effect of i.v. MgSO4.Hospital admission: Mg...
1,"Tiffany (12), 1993, Detroit, United States","RCT, double blind",I.v. MgSO4 (as a bolus and/or infusion) as adj...,PEF and FEV1(260 min),"48 patients(F: 56%). MgSO4 bolus: 15, MgSO4 in...","SoC: nebulized albuterol, i.v. steroids, amino...",No effect of i.v. MgSO4.FEV1: p = 0.96.PEF: p ...
2,"Bloch (10), 1995, New York, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,"Hospital admission (4 h), FEV1 (120 min)","135 patients (F: 72%). MgSO4: 67, control: 68....","SoC: nebulized albuterol, i.v. steroids (FEV1 ...","Effect on patients with severe exacerbations, ..."
3,"Boonyavorakul (13), 2,000, Thailand","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,Hospital admission(240 min),"33 patients(F: 88%). MgSO4: 17, control: 16.Ac...","SoC: i.v. steroids, nebulized salbutamol, O2 i...",No effect of i.v. MgSO4.Hospital admission:MgS...
4,"Porter (14), 2001, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,1: PEF2: hospital admission(60 min),"42 patients(F: 64%). MgSO4: 18, control: 24. A...","SoC: i.v. steroids, nebulized albuterol, O2 if...",No effect of i.v. MgSO4.PEF:MgSO4: 211 ± 104 L...
5,"Silverman (15), 2002, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,"1: FEV1%p2: hospital admission, PEF(240 min)","240 patients (120 + 120),(F: 42%)Acute asthma ...","SoC: i.v. steroids, nebulized albuterol, O2.Mg...",Effect of i.v. MgSO4 on FEV1 and PEF.FEV1%p:Mg...
6,"Bradshaw (16), 2007, United Kingdom","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,1: PEF%p2: hospital admission(60 min),"129 patients(F: 57%). MgSO4: 62, control: 67.A...","SoC: 35% O2, i.v. steroids, nebulized salbutam...","No effect of i.v. MgSO4.PEF%p: MgSO4: 65.4, co..."
7,"Singh (8), 2008, India","RCT, single blind",I.v. MgSO4 as adjunct to standard therapy,"FEV1%p, hospital admission(120 min)","60 patients (30 + 30),(F: 52%)Acute asthma (FE...","SoC: i.v. steroids, nebulized salbutamol + ipr...",Effect of i.v. MgSO4.ΔFEV1%p:MgSO4: 40.77 ± 9....
8,"Goodacre (9), 2014, United Kingdom","RCT, double blind",I.v. or nebulized MgSO4 as adjunct to standard...,1: hospital admission2: PEF,"1,084 patients(F: 70%).i.v. MgSO4: 394, placeb...","SoC: Differing (O2, steroids, salbutamol, ipra...",No effect of i.v. or nebulized MgSO4.ΔPEF:I.v....
9,"Nannini (17), 2,000, Argentina","RCT, double blind",Nebulized MgSO4 as adjunct to salbutamol,1: PEF(20 min),"35 patients.(F: 33%)MgSO4: 19, control: 16.Acu...",SoC: nebulized salbutamolMgSO4: nebulized 3 ml...,Effect of nebulized MgSO4.ΔPEF:MgSO4: 134 ± 70...


## Advanced Usage Examples


In [29]:
# Example 1: Search for papers by source
pmc_papers = reader.search_papers('pmc', field='source')
print(f"PMC papers: {len(pmc_papers)}")
display(pmc_papers[['paper_id', 'source', 'num_tables', 'num_images']])


PMC papers: 13


,paper_id,source,num_tables,num_images
0,PMC10420062,pmc,7,5
1,PMC9640961,pmc,10,3
2,PMC10848175,pmc,3,15
3,PMC8369948,pmc,4,9
4,PMC11370728,pmc,3,10
5,PMC7889465,pmc,4,8
6,PMC8467008,pmc,6,5
7,PMC8588837,pmc,6,6
8,PMC10099188,pmc,4,12
9,PMC8597000,pmc,2,16


In [30]:
# Example 2: View a complete document
if len(df_documents) > 0:
    first_paper_id = df_documents.iloc[0]['paper_id']
    print(f"Viewing document: {first_paper_id}\n")
    show_document(first_paper_id)


Viewing document: PMC10420062

Paper ID: PMC10420062
Source: pmc
PMCID: PMC10420062
Authors: Alma Holm Rovsing, Osman Savran, Charlotte Suppli Ulrik
Emails: 0 found
Files: MD=True, HTML=True, PDF=True
Content: 10 sections, 7 tables, 5 images



In [31]:
# Example 3: Author summary
df_authors = reader.get_authors_summary()
print(f"Total unique authors: {len(df_authors)}")
print("\nMost prolific authors:")
display(df_authors.sort_values('num_papers', ascending=False).head(10))


Total unique authors: 75

Most prolific authors:


,author,num_papers,papers
2,Antonella Muraro,2,"[PMC10848175, PMC10099188]"
5,Athina Ioannidou,2,"[PMC10848175, PMC10099188]"
6,Aziz Sheikh,2,"[PMC10848175, PMC10099188]"
9,Berber Vlieg‐Boerstra,2,"[PMC10848175, PMC10099188]"
25,Graham Roberts,2,"[PMC10848175, PMC10099188]"
24,Graciela Rovner,2,"[PMC10848175, PMC10099188]"
23,Giulia C. I. Spolidoro,2,"[PMC10848175, PMC10099188]"
21,Ekaterina Khaleva,2,"[PMC10848175, PMC10099188]"
18,Daniil Lisik,2,"[PMC10848175, PMC10099188]"
13,Carina Venter,2,"[PMC10848175, PMC10099188]"


In [32]:
# Example 4: View all sections of a document (tables + images)
if len(df_documents) > 0:
    first_paper_id = df_documents.iloc[0]['paper_id']
    print(f"Viewing all sections of: {first_paper_id}\n")
    show_document_sections(first_paper_id)


Viewing all sections of: PMC10420062


Section 1: Introduction

Section 2: Methods

Section 3: Results

📊 Tables (4):

  Table 1 (index: 0):


,Study,Design,Objective,Outcomes,Patients,Treatment,Result
0,"Green (11), 1992, California, United States",RCT,I.v. MgSO4 as adjunct to standard therapy,Hospital admission and PEF.,120 patients (F: 77%). MgSO4: 58. Control: 62....,"SoC: nebulized albuterol, i.v. steroids, oxyge...",No effect of i.v. MgSO4.Hospital admission: Mg...
1,"Tiffany (12), 1993, Detroit, United States","RCT, double blind",I.v. MgSO4 (as a bolus and/or infusion) as adj...,PEF and FEV1(260 min),"48 patients(F: 56%). MgSO4 bolus: 15, MgSO4 in...","SoC: nebulized albuterol, i.v. steroids, amino...",No effect of i.v. MgSO4.FEV1: p = 0.96.PEF: p ...
2,"Bloch (10), 1995, New York, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,"Hospital admission (4 h), FEV1 (120 min)","135 patients (F: 72%). MgSO4: 67, control: 68....","SoC: nebulized albuterol, i.v. steroids (FEV1 ...","Effect on patients with severe exacerbations, ..."
3,"Boonyavorakul (13), 2,000, Thailand","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,Hospital admission(240 min),"33 patients(F: 88%). MgSO4: 17, control: 16.Ac...","SoC: i.v. steroids, nebulized salbutamol, O2 i...",No effect of i.v. MgSO4.Hospital admission:MgS...
4,"Porter (14), 2001, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,1: PEF2: hospital admission(60 min),"42 patients(F: 64%). MgSO4: 18, control: 24. A...","SoC: i.v. steroids, nebulized albuterol, O2 if...",No effect of i.v. MgSO4.PEF:MgSO4: 211 ± 104 L...
5,"Silverman (15), 2002, United States","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,"1: FEV1%p2: hospital admission, PEF(240 min)","240 patients (120 + 120),(F: 42%)Acute asthma ...","SoC: i.v. steroids, nebulized albuterol, O2.Mg...",Effect of i.v. MgSO4 on FEV1 and PEF.FEV1%p:Mg...
6,"Bradshaw (16), 2007, United Kingdom","RCT, double blind",I.v. MgSO4 as adjunct to standard therapy,1: PEF%p2: hospital admission(60 min),"129 patients(F: 57%). MgSO4: 62, control: 67.A...","SoC: 35% O2, i.v. steroids, nebulized salbutam...","No effect of i.v. MgSO4.PEF%p: MgSO4: 65.4, co..."
7,"Singh (8), 2008, India","RCT, single blind",I.v. MgSO4 as adjunct to standard therapy,"FEV1%p, hospital admission(120 min)","60 patients (30 + 30),(F: 52%)Acute asthma (FE...","SoC: i.v. steroids, nebulized salbutamol + ipr...",Effect of i.v. MgSO4.ΔFEV1%p:MgSO4: 40.77 ± 9....
8,"Goodacre (9), 2014, United Kingdom","RCT, double blind",I.v. or nebulized MgSO4 as adjunct to standard...,1: hospital admission2: PEF,"1,084 patients(F: 70%).i.v. MgSO4: 394, placeb...","SoC: Differing (O2, steroids, salbutamol, ipra...",No effect of i.v. or nebulized MgSO4.ΔPEF:I.v....
9,"Nannini (17), 2,000, Argentina","RCT, double blind",Nebulized MgSO4 as adjunct to salbutamol,1: PEF(20 min),"35 patients.(F: 33%)MgSO4: 19, control: 16.Acu...",SoC: nebulized salbutamolMgSO4: nebulized 3 ml...,Effect of nebulized MgSO4.ΔPEF:MgSO4: 134 ± 70...



  Table 2 (index: 1):


,Randomized controlled trial omitted,MD (95% CI),Heterogeneity
0,(A),NaN,NaN
1,Porter et al. (16),11.30 (−11.23–33.83),"I2 = 47%, t2 = 193.8484, p = 0.15"
2,Silverman et al. (11),−3.37 (−28.18–21.43),"I2 = 35%, t2 = 195.7486, p = 0.21"
3,Goodacre et al. (10),−1.27 (−46.21–43.68),"I2 = 67%, t2 = 1040.1831, p = 0.05"
4,Green et al. (12),10.42 (−18.95–39.78),"I2 = 57%, t2 = 377.1589, p = 0.10"
5,Total,5.49 (−18.67–29.65),"I2 = 51%, t2 = 296.8264, p = 0.10"
6,(B),NaN,NaN
7,Nannini et al. (20),21.00 (−6.74–48.74),"I2 = 93%, t2 = 553.4887, p < 0.01"
8,Ahmed et al. (22),9.66 (−14.72–34.03),"I2 = 80%, t2 = 301.8074, p < 0.01"
9,Goodacre et al. (10),37.06 (5.29–68.83),"I2 = 89%, t2 = 567.8665, p < 0.01"



  Table 3 (index: 2):


...


Section 8: Quality and bias risk assessment

📊 Tables (2):

  Table 1 (index: 0):


,Study,Risk of bias from the randomization process,Risk of bias due to deviations from the intended interventions,Risk of bias due to missing outcome data,Risk of bias in measurement of the outcome,Risk of bias in selection of the reported results,Overall risk of bias
0,Green,High risk,High risk,Low risk,Some risk,Some risk,High risk
1,Tiffany,Some risk,Low risk,Low risk,High risk,High risk,High risk
2,Bloch,Low risk,Low risk,Low risk,Low risk,High risk,High risk
3,Boonyavorakul,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk
4,Porter,Low risk,Low risk,Low risk,Low risk,Some risk,Low risk
5,Silverman,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk
6,Bradshaw,Low risk,Low risk,Low risk,Low risk,Some risk,Some concerns
7,Singh,Low risk,Low risk,Some risk,Low risk,Low risk,Low risk
8,Goodacre,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk



  Table 2 (index: 1):


,Study,Risk of bias from the randomization process,Risk of bias due to deviations from the intended interventions,Risk of bias due to missing outcome data,Risk of bias in measurement of the outcome,Risk of bias in selection of the reported results,Overall risk of bias
0,Nannini,Some risk,Low risk,High risk,Low risk,High risk,High risk
1,Bessmertny,Low risk,Low risk,Some risk,Low risk,High risk,High risk
2,Hughes,Low risk,Low risk,Low risk,Low risk,High risk,High risk
3,Kokturk,Some risk,Some risk,Low risk,High risk,Low risk,High risk
4,Gallegos-Solórzano,Some risk,Low risk,Low risk,Low risk,Low risk,Some concerns
5,Ahmed,Some risk,High risk,High risk,High risk,High risk,High risk
6,Goodacre,Low risk,Low risk,Low risk,Low risk,Low risk,Low risk
7,Badawy,Some risk,High risk,High risk,Low risk,High risk,High risk
8,Hossein,Low risk,Low risk,Some risk,Low risk,Some risk,Some concerns



Section 9: Discussion

Section 10: Conclusion


## Using Helper Functions

Use the helper functions below to inspect specific tables and images:

- `show_table(index)` — Display a specific table by index
- `show_image(index)` — Display a specific image by index
- `show_document(paper_id)` — Show metadata for a paper
- `show_document_sections(paper_id)` — Display all sections with tables and images

Examples:
```python
show_table(0)           # First table
show_image(0)           # First image
show_table(5)           # 6th table
show_image(3)           # 4th image
```
